In [1]:
import pandas as pd

df = pd.read_csv("salaries.csv")

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3755 entries, 0 to 3754
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   work_year           3755 non-null   int64 
 1   experience_level    3755 non-null   object
 2   employment_type     3755 non-null   object
 3   job_title           3755 non-null   object
 4   salary              3755 non-null   int64 
 5   salary_currency     3755 non-null   object
 6   salary_in_usd       3755 non-null   int64 
 7   employee_residence  3755 non-null   object
 8   remote_ratio        3755 non-null   int64 
 9   company_location    3755 non-null   object
 10  company_size        3755 non-null   object
dtypes: int64(4), object(7)
memory usage: 322.8+ KB
None


In [2]:
df = df[df["employment_type"] == "FT"]
df.shape

(3718, 11)

In [3]:
df = df[df["work_year"] >= 2023]
df.shape

(1780, 11)

In [4]:
df = df[[
    "work_year",
    "job_title",
    "experience_level",
    "salary_in_usd",
    "company_location",
    "employee_residence",
    "company_size",
    "remote_ratio"
]]

In [5]:
df.head()

,work_year,job_title,experience_level,salary_in_usd,company_location,employee_residence,company_size,remote_ratio
0,2023,Principal Data Scientist,SE,85847,ES,ES,L,100
3,2023,Data Scientist,SE,175000,CA,CA,M,100
4,2023,Data Scientist,SE,120000,CA,CA,M,100
5,2023,Applied Scientist,SE,222200,US,US,L,0
6,2023,Applied Scientist,SE,136000,US,US,L,0


In [6]:
df["experience_level"] = df["experience_level"].replace({
    "EN": "Entry",
    "MI": "Mid",
    "SE": "Senior",
    "EX": "Executive"
})

In [7]:
df["job_title"].value_counts().head(15)

job_title
Data Engineer                499
Data Scientist               370
Data Analyst                 306
Machine Learning Engineer    158
Research Scientist            55
Data Architect                52
Analytics Engineer            46
Applied Scientist             40
Research Engineer             32
Data Manager                  22
Data Science Manager          22
Data Analytics Manager        12
ML Engineer                   12
BI Developer                  10
Computer Vision Engineer      10
Name: count, dtype: int64

In [8]:
benchmark_roles = [
    "Data Engineer",
    "Data Scientist",
    "Data Analyst",
    "Machine Learning Engineer",
    "Research Scientist",
    "Data Architect"
]

df_benchmark = df[df["job_title"].isin(benchmark_roles)]
df_benchmark.shape

(1440, 8)

In [9]:
benchmark_table = (
    df_benchmark
    .groupby("job_title")["salary_in_usd"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
)

benchmark_table.columns = ["P25", "Median", "P75"]
benchmark_table = benchmark_table.round(0)

benchmark_table

,P25,Median,P75
job_title,,,
Data Analyst,80000.0,107400.0,142750.0
Data Architect,114750.0,159750.0,180000.0
Data Engineer,115000.0,144000.0,180000.0
Data Scientist,120000.0,155500.0,190000.0
Machine Learning Engineer,128710.0,151769.0,203963.0
Research Scientist,145450.0,169200.0,210000.0


In [10]:
level_benchmark = (
    df_benchmark
    .groupby(["job_title", "experience_level"])["salary_in_usd"]
    .median()
    .unstack()
)

level_benchmark.round(0)

experience_level,Entry,Executive,Mid,Senior
job_title,,,,
Data Analyst,75000.0,NaN,100000.0,121600.0
Data Architect,NaN,167500.0,137000.0,159750.0
Data Engineer,85000.0,208500.0,120000.0,150000.0
Data Scientist,73224.0,192500.0,100000.0,165000.0
Machine Learning Engineer,154540.0,NaN,130000.0,163800.0
Research Scientist,150000.0,NaN,150900.0,190000.0


In [11]:
benchmark_table.to_csv("salary_market_benchmark.csv")
level_benchmark.to_csv("salary_by_experience.csv")

In [12]:
benchmark_table

,P25,Median,P75
job_title,,,
Data Analyst,80000.0,107400.0,142750.0
Data Architect,114750.0,159750.0,180000.0
Data Engineer,115000.0,144000.0,180000.0
Data Scientist,120000.0,155500.0,190000.0
Machine Learning Engineer,128710.0,151769.0,203963.0
Research Scientist,145450.0,169200.0,210000.0


In [14]:
benchmark_table = (
    df_benchmark
    .groupby("job_title")["salary_in_usd"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
)

benchmark_table.columns = ["P25", "Median", "P75"]
benchmark_table = benchmark_table.round(0)

benchmark_table

,P25,Median,P75
job_title,,,
Data Analyst,80000.0,107400.0,142750.0
Data Architect,114750.0,159750.0,180000.0
Data Engineer,115000.0,144000.0,180000.0
Data Scientist,120000.0,155500.0,190000.0
Machine Learning Engineer,128710.0,151769.0,203963.0
Research Scientist,145450.0,169200.0,210000.0


In [15]:
company_size_pay = (
    df_benchmark
    .groupby("company_size")["salary_in_usd"]
    .median()
)

company_size_pay

company_size
L    105700.0
M    145000.0
S     43000.0
Name: salary_in_usd, dtype: float64

In [16]:
benchmark_table.to_csv("salary_percentiles.csv")
level_benchmark.to_csv("salary_by_experience.csv")
company_size_pay.to_csv("salary_by_company_size.csv")

In [17]:
company_salary = pd.DataFrame({
    "job_title": [
        "Data Analyst",
        "Data Engineer",
        "Data Scientist",
        "Machine Learning Engineer"
    ],
    "company_salary": [
        100000,
        140000,
        150000,
        145000
    ]
})

In [19]:
market_position = benchmark_table.merge(company_salary, on="job_title")
market_position["position_vs_market"] = (
    market_position["company_salary"] /
    market_position["Median"]
)

In [20]:
market_position.to_csv("market_positioning.csv")